# Compute `FCEN_LYA` for a catalogue

Adds `FCEN_LYA` (central flux fraction within ±`v_window` km/s of systemic)  
and `FCEN_LYA_ERR` (1-σ MC uncertainty) as new columns to a FITS catalogue,  
then writes the result to a new file.

**Before running:** set `CATALOGUE_IN` and `CATALOGUE_OUT` in cell 2.

In [ ]:
# ── USER SETTINGS ────────────────────────────────────────────────────────────
CATALOGUE_IN  = "../megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_weight_skysub.fits"   # input FITS catalogue
CATALOGUE_OUT = "../megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_fcen_weight_skysub.fits"  # output path

V_WINDOW = 100.0    # km/s half-width of the central velocity window
V_RANGE  = 3000.0   # km/s half-width used for the total-flux integral
N_GRID   = 2000     # velocity grid points (see benchmarks in README)
N_MC     = 500      # Monte Carlo draws for uncertainty; set 0 to skip
SEED     = 42       # RNG seed for reproducibility
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
from astropy.table import Table, Column
from tqdm.auto import tqdm
from tangelo.spectroscopy import lya_central_flux_fraction

In [ ]:
cat = Table.read(CATALOGUE_IN)
print(f"Loaded {len(cat)} rows from {CATALOGUE_IN}")

In [ ]:
rng = np.random.default_rng(SEED)

fracs = np.full(len(cat), np.nan)
errs  = np.full(len(cat), np.nan)

for i, row in enumerate(tqdm(cat, desc="FCEN_LYA")):
    try:
        fracs[i], errs[i] = lya_central_flux_fraction(
            row,
            v_window=V_WINDOW,
            v_range=V_RANGE,
            n_grid=N_GRID,
            n_mc=N_MC,
            rng=rng,
        )
    except (KeyError, ValueError, TypeError):
        pass  # leaves NaN for rows that lack required fit parameters

n_good = np.sum(np.isfinite(fracs))
print(f"Computed fraction for {n_good}/{len(cat)} rows  "
      f"({len(cat) - n_good} skipped / NaN)")

In [ ]:
# Replace existing columns if a previous run already added them
for colname, data in [("FCEN_LYA", fracs), ("FCEN_LYA_ERR", errs)]:
    if colname in cat.colnames:
        cat.replace_column(colname, Column(data, name=colname))
    else:
        cat.add_column(Column(data, name=colname))

cat.write(CATALOGUE_OUT, overwrite=True)
print(f"Saved to {CATALOGUE_OUT}")

In [ ]:
# Quick sanity check
import matplotlib.pyplot as plt

finite = np.isfinite(fracs)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(fracs[finite], bins=40)
ax.set_xlabel(r"$f_{\rm cen}$ (Ly$\alpha$ central flux fraction)")
ax.set_ylabel("N")
ax.set_title(f"v_window = {V_WINDOW} km/s  |  {n_good} objects")
plt.tight_layout()
plt.show()